# Bildverarbeitung Grundlagen

## Einführung

Willkommen zu dieser praktischen Übung zur Bildverarbeitung! In diesem Notebook werden wir grundlegende Techniken der Bildmanipulation von Grund auf implementieren. Das Ziel ist es, ein tiefes Verständnis für die zugrunde liegenden Operationen zu entwickeln, anstatt nur fertige Bibliotheksfunktionen zu verwenden.

Wir werden drei Hauptthemen behandeln:

1. **Farbraumtransformation**: Wir laden ein Farbbild und wandeln es in den HSV-Farbraum um. Dabei lernst du, wie du interaktive Steuerelemente (Slider) einsetzt, um Bildparameter wie Farbton, Sättigung und Helligkeit in Echtzeit zu verändern.

2. **Grauwertkonvertierung und Quantisierung**: Wir wandeln das Farbbild in ein Graustufenbild um und reduzieren anschließend die Anzahl der Helligkeitsstufen (Quantisierung).

3. **Fehlerdiffusion**: Zum Schluss implementieren wir den Floyd-Steinberg-Algorithmus zur Fehlerdiffusion. Dieses Verfahren erzeugt eine hochwertigere Darstellung von Graustufenbildern mit einer begrenzten Anzahl von Farben, indem es den Quantisierungsfehler auf benachbarte Pixel verteilt.

Für die Bearbeitung benötigst du die Bibliotheken `Pillow`, `numpy`, `matplotlib` und `ipywidgets`

---

### Importieren der benötigten Bibliotheken

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from ipywidgets import interact, fixed

Laden des Beispielbildes

In [ ]:
barboon_rgb = Image.open("Baboon.png").convert("RGB")
barboon_rgb

## Teilaufgabe 1: Farbraumtransformation (RGB zu HSV und zurück)
In dieser Aufgabe wandeln wir ein Bild vom RGB- in den HSV-Farbraum um und wieder zurück. 
HSV (Hue, Saturation, Value) ist oft intuitiver für die Farbanpassung. 
Wir werden die Konvertierungen manuell mit einer Schleife über alle Pixel implementieren, um den Prozess nachzuvollziehen.
Die mathematischen Formeln für die Umwandlung findest du u.a. [hier](https://de.wikipedia.org/wiki/HSV-Farbraum#Umrechnung_von_RGB_in_HSV_und_umgekehrt).

Achte darauf:
- dass die Farbwerte in RGB im Bereich [0, 255] und in HSV im Bereich H: [0, 360], S: [0, 1], V: [0, 1] liegen.
- dass das RGB Bild vor der Transformation in einen np.float32 Array umgewandelt (ndarray.astype()) und die Werte auf [0, 1] skaliert werden.

```python

In [ ]:
rgb_img = np.array(barboon_rgb.copy())
rgb_img.shape

In [ ]:
def pixel_rgb_to_hsv(r: int, g: int, b: int) -> tuple[float, float, float]:
    """Converts a pixel with RGB values to HSV.

    Attributes:
        r: Red in range [0, 1]
        g: Green in range [0, 1]
        b: Blue in range [0, 1]

    Returns:
        h: Hue [0, 360)
        s: Saturation [0, 1]
        v: Value [0, 1]
    """
    c_max = max(r, g, b)
    c_min = min(r, g, b)
    delta = c_max - c_min

    v = c_max

    if c_max == 0:
        s = 0.0
    else:
        s = delta / c_max

    if delta == 0:
        h = 0.0
    elif c_max == r:
        h = 60.0 * (((g - b) / delta) % 6)
    elif c_max == g:
        h = 60.0 * (((b - r) / delta) + 2)
    else:
        h = 60.0 * (((r - g) / delta) + 4)

    return h, s, v


def pixel_hsv_to_rgb(h: float, s: float, v: float) -> tuple[int, int, int]:
    """Converts a pixel with HSV values to RGB.

    Attributes:
        h: Hue in range [0, 360)
        s: Saturation in range [0, 1]
        v: Value in range [0, 1]

    Returns:
        r, g, b: Each in [0, 1]
    """
    c = v * s
    x = c * (1 - abs((h / 60) % 2 - 1))
    m = v - c

    if 0 <= h < 60:
        r1, g1, b1 = c, x, 0
    elif 60 <= h < 120:
        r1, g1, b1 = x, c, 0
    elif 120 <= h < 180:
        r1, g1, b1 = 0, c, x
    elif 180 <= h < 240:
        r1, g1, b1 = 0, x, c
    elif 240 <= h < 300:
        r1, g1, b1 = x, 0, c
    else:
        r1, g1, b1 = c, 0, x

    return r1 + m, g1 + m, b1 + m


def convert_rgb_to_hsv_loop(rgb_img: np.ndarray) -> np.ndarray:
    """Convert an RGB image to HSV using a loop over all pixels.

    Attributes:
        rgb_img: Input RGB image as a NumPy array of shape (H, W, 3) with values in [0, 255].

    Returns:
        hsv_img: Output HSV image as a NumPy array of shape (H, W, 3) with H in [0, 360), S and V in [0, 1].

    """
    height, width = rgb_img.shape[:2]
    hsv_img = np.zeros((height, width, 3), dtype=np.float32)

    img_float = rgb_img.astype(np.float32) / 255.0

    for y in range(height):
        for x in range(width):
            r, g, b = img_float[y, x]
            h, s, v = pixel_rgb_to_hsv(r, g, b)
            hsv_img[y, x] = [h, s, v]

    return hsv_img


def convert_hsv_to_rgb_loop(hsv_img: np.ndarray) -> np.ndarray:
    """Convert an HSV image to RGB using a loop over all pixels.

    Attributes:
        hsv_img: Input HSV image as a NumPy array of shape (H, W, 3) with H in [0, 360), S and V in [0, 1].
    """
    height, width = hsv_img.shape[:2]
    rgb_img = np.zeros((height, width, 3), dtype=np.uint8)

    for y in range(height):
        for x in range(width):
            h, s, v = hsv_img[y, x]
            r, g, b = pixel_hsv_to_rgb(h, s, v)
            rgb_img[y, x] = [r * 255, g * 255, b * 255]

    return rgb_img


def adjust_hsv_and_display(hsv_img: np.ndarray, hue_shift: float, saturation_scale: float, value_scale: float) -> None:
    """Adjust HSV values and display the resulting image.

    Attributes:
        hsv_img: Input HSV image as a NumPy array of shape (H, W, 3) with H in [0, 360), S and V in [0, 1].
        hue_shift: Amount to shift the hue channel, range [0, 360) degrees.
        saturation_scale: Factor to scale the saturation channel, range [0, 1].
        value_scale: Factor to scale the value channel, range [0, 1].
    """
    hsv_adjusted = hsv_img.copy()

    hsv_adjusted[:, :, 0] = (hsv_adjusted[:, :, 0] + hue_shift) % 360

    hsv_adjusted[:, :, 1] = np.clip(hsv_adjusted[:, :, 1] * saturation_scale, 0, 1)

    hsv_adjusted[:, :, 2] = np.clip(hsv_adjusted[:, :, 2] * value_scale, 0, 1)

    rgb_result = convert_hsv_to_rgb_loop(hsv_adjusted)
    display(Image.fromarray(rgb_result))

In [ ]:
fix, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(rgb_img)
ax[0].set_title("Original RGB Bild")
ax[0].axis('off')

hsv_img = convert_rgb_to_hsv_loop(rgb_img)
hsv_to_rgb = convert_hsv_to_rgb_loop(hsv_img)

ax[1].imshow(hsv_to_rgb)
ax[1].set_title("RGB Bild nach RGB->HSV")
ax[1].axis('off')
plt.show()

In [ ]:
image_hsv = convert_rgb_to_hsv_loop(rgb_img)
interact(
    adjust_hsv_and_display,
    hsv_img=fixed(image_hsv),
    hue_shift=widgets.FloatSlider(min=0.0, max=360.0, step=1.0, value=0, description='Farbton:'),
    saturation_scale=widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=1, description='Sättigung:'),
    value_scale=widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=1, description='Helligkeit:')
);

## Teilaufgabe 2: Grauwertkonvertierung und Quantisierung

In diesem Teil wandeln wir das Farbbild in ein Graustufenbild um und reduzieren anschließend die Anzahl der verfügbaren Helligkeitswerte. Diesen Prozess nennt man Quantisierung.

### 2.1 Manuelle Grauwertkonvertierung

**Deine Aufgabe**: Implementiere eine Funktion, die ein RGB-Bild in ein Graustufenbild umwandelt. Verwende die Leuchtdichte-Formel: `Grauwert = 0.299 * R + 0.587 * G + 0.114 * B`.



In [ ]:
def convert_to_grayscale_loop(rgb_img: np.ndarray) -> np.ndarray:
    """Convert an RGB image to grayscale using a loop over all pixels.

    Attributes:
        rgb_img: Input RGB image as a NumPy array of shape (H, W, 3) with values in [0, 255].
    """
    height, width = rgb_img.shape[:2]
    gray_img = np.zeros((height, width), dtype=np.float32)

    for y in range(height):
        for x in range(width):
            r, g, b = rgb_img[y, x]
            gray_img[y, x] = 0.299 * r + 0.587 * g + 0.114 * b

    return gray_img

In [ ]:
rgb_img = np.array(barboon_rgb.copy())
image_gray = convert_to_grayscale_loop(rgb_img)

plt.figure(figsize=(6, 6))
plt.imshow(image_gray, vmin=0, vmax=255, cmap='gray')
plt.title("Graustufenbild")
plt.axis('off')
plt.show()

## 2.2 Quantisierung

**Deine Aufgabe**: Implementiere die Funktion `quantize_grayscale`, die ein Graustufenbild auf eine bestimmte Anzahl von Helligkeitsstufen (num_intensities) reduziert.

Die Funktion zur Quantisierung `Q(p)`eines Grauwertpixels `p` in `S` Stufen ist:

$$Q(p) = \Delta \cdot \left\lfloor \frac{p}{\Delta} \right\rfloor$$

wobei $$\Delta = 256 / S$$ die Schrittweite zwischen den Stufen ist.

In [ ]:
def quantize_grayscale(gray_img: np.ndarray, num_intensities: int) -> None:
    """Quantize a grayscale image to a specified number of levels.

    Attributes:
        gray_img: Input grayscale image as a NumPy array of shape (H, W) with values in [0, 255].
        num_intensities: Number of intensity levels to quantize to (e.g., 2, 4, 8, 16, 32, 64, 128, 256).
    """
    gray_quantized = np.zeros_like(gray_img, dtype=np.float32)

    delta = 256 / num_intensities

    height, width = gray_img.shape
    for y in range(height):
        for x in range(width):
            p = gray_img[y, x]
            gray_quantized[y, x] = delta * int(p / delta)

    display(Image.fromarray(gray_quantized.astype(np.uint8)).resize((420, 420)))

In [ ]:
rgb_img = np.array(barboon_rgb.copy())
image_gray = convert_to_grayscale_loop(rgb_img)

interact(
    quantize_grayscale,
    gray_img=fixed(image_gray),
    num_intensities=widgets.IntSlider(
        min=2,
        max=64,
        step=2,
        value=8,
        description='Anzahl Stufen:'
    )
);

## Teilaufgabe 3: Fehlerdiffusion

Die einfache Quantisierung kann zu sichtbaren Kanten (Banding) führen. Fehlerdiffusion ist eine Technik, die diesen Effekt reduziert, indem der Quantisierungsfehler eines Pixels auf seine Nachbarn verteilt wird. Wir implementieren den [Floyd-Steinberg-Algorithmus](https://de.wikipedia.org/wiki/Floyd-Steinberg-Algorithmus).

**Deine Aufgabe**: Implementiere die Funktion floyd_steinberg_diffusion.

Tipp:
- Im Pseudo-Code des Wikipedia-Artkels wird find_closest_palette_color verwendet, um den nächstgelegenen Farbwert zu finden. In unserem Fall kannst du die Quantisierungsformel aus Teilaufgabe 2.2 verwenden.

In [ ]:
def floyd_steinberg_diffusion(gray_img: np.ndarray, num_intensities: int) -> None:
    img = gray_img.astype(np.float32).copy()
    height, width = img.shape

    delta = 256 / num_intensities

    for y in range(height):
        for x in range(width):
            old_pixel = img[y, x]
            new_pixel = np.clip(delta * int(old_pixel / delta), 0, 255)
            img[y, x] = new_pixel

            error = old_pixel - new_pixel

            if x + 1 < width:
                img[y, x + 1] = np.clip(img[y, x + 1] + error * 7 / 16, 0, 255)
            if y + 1 < height:
                if x - 1 >= 0:
                    img[y + 1, x - 1] = np.clip(img[y + 1, x - 1] + error * 3 / 16, 0, 255)
                img[y + 1, x] = np.clip(img[y + 1, x] + error * 5 / 16, 0, 255)
                if x + 1 < width:
                    img[y + 1, x + 1] = np.clip(img[y + 1, x + 1] + error * 1 / 16, 0, 255)

    display(Image.fromarray(img.astype(np.uint8)).resize((420, 420)))

In [ ]:
rgb_img = np.array(barboon_rgb.copy())
image_gray = convert_to_grayscale_loop(rgb_img)

interact(
    floyd_steinberg_diffusion,
    gray_img=fixed(image_gray),
    num_intensities=widgets.IntSlider(
        min=2,
        max=64,
        step=2,
        value=8,
        description='Anzahl Stufen:'
    )
);